# کارخانه داده — NYC 311 (Term 3 · Stage 1)

این نوتبوک **روایت** کارخانه است، نه جای اجرای آن. کل مسیر داده —
دریافت، قرنطینه، پاک‌سازی، join با منبع دوم و سه خروجی — در پکیج
`data_factory` اجرا می‌شود (`.venv/bin/data-factory run`)، و اینجا فقط
خروجیِ منتشرشده را می‌خوانیم و درباره‌اش استدلال می‌کنیم.

**چرا این تفکیک؟** اگر تحلیل داخل نوتبوک باشد، هر بار اجرا یک نسخه‌ی
دیگر از داده می‌سازد و «قابل بازتولید» بی‌معنا می‌شود؛ اینجا نوتبوک
هیچ فایلی نمی‌نویسد و هیچ فرضی درباره‌ی داده‌ی خام ندارد.

**پیش‌نیاز اجرا:** `data-factory run` (یا `--since-last-run` برای پنجره‌ی
متحرک) و وجود `data/reference/zip_population.csv`.

**همه‌ی منطق تحلیل** در `src/data_factory/eda.py` است و تست آفلاین دارد
(`tests/test_eda.py`)؛ هر عددی که در ادامه می‌بینید از همان توابع می‌آید.

> قرارداد: هیچ عددی دست‌نویس نیست. هر جمله‌ی نتیجه در سلول‌های کد از روی
> داده ساخته می‌شود، پس با تغییر داده، جمله هم عوض می‌شود.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data_factory import eda
from data_factory.manifest import sha256_file
from data_factory.population import load_population

SEED = 20261024  # ثابت و یک‌جا (بخش ۵.۵ قرارداد)
rng = np.random.default_rng(SEED)

DATA = Path("data")  # مسیر نسبی، وابسته به ریشه‌ی پروژه
CASES = DATA / "curated" / "cases.csv"
QUALITY = DATA / "curated" / "data_quality.csv"
MANIFEST = DATA / "curated" / "run_manifest.json"
POPULATION = DATA / "reference" / "zip_population.csv"

missing = [str(path) for path in (CASES, QUALITY, MANIFEST, POPULATION) if not path.is_file()]
assert not missing, (
    "ابتدا کارخانه را اجرا کن: data-factory run — فایل(های) غایب: " + ", ".join(missing)
)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
print("pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)

## ۱) این داده چیست و از کجا آمده؟

قبل از هر ادعایی باید مطمئن شویم فایلی که می‌خوانیم *همان* فایلی است که
کارخانه منتشر کرده. سه شاهد:

1. `run_manifest.json` — پنجره‌ی دریافت‌شده، معیار محدودسازی (source، agency،
   فهرست صریح انواع)، تعداد رکورد خام/پاس‌شده/قرنطینه و بودجه‌ی درخواست.
2. **hash** فایل `cases.csv` در manifest؛ همان hash را روی فایل واقعی
   دوباره حساب می‌کنیم.
3. جدول کیفیت تولیدشده (`data_quality.csv`) که بخش بعدی می‌خواند.

اگر این سه با هم بخوانند، تحلیل روی همان داده‌ای است که گزارش به آن ارجاع
می‌دهد.

In [ ]:
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))

window = manifest["window"]
scope = manifest["scope"]
fetch = manifest["fetch"]
print(f"window      : {window['start']} .. {window['end']}  (mode: {window['mode']})")
print(f"scope       : {scope['agency']} / {len(scope['complaint_types'])} complaint types")
print(f"records     : raw {fetch['records_raw']:,} | passed {manifest['passed_records']:,}"
      f" | quarantined {manifest['quarantine']['records']:,}")
print(f"fetch cost  : {fetch['http_requests']:,} HTTP requests | {fetch['batches']:,} batches"
      f" | peak in memory {fetch['peak_in_memory_records']:,} records")
print(f"archived    : {manifest['archived_windows'] or 'none'}")
print("outputs     :")
for name, entry in sorted(manifest["outputs"].items()):
    print(f"  {name:18s} {entry['sha256'][:12]}…  {entry['bytes']:>12,} bytes")

In [ ]:
frame = eda.load_cases(CASES)
population = load_population(POPULATION)

case_hash = sha256_file(CASES)
assert case_hash == manifest["outputs"]["cases.csv"]["sha256"], (
    "cases.csv با hash ثبت‌شده در manifest نمی‌خواند — کارخانه را دوباره اجرا کن"
)
print(f"{len(frame):,} rows × {frame.shape[1]} columns (۱۰ ستون قرارداد + ستون‌های مشتق‌شده)")
frame[["unique_key", "created_date", "closed_date", "complaint_type",
       "borough", "population", "duration_minutes"]].head(3)

## ۲) کیفیت داده — گزارشِ تولیدشده، نه دست‌نویس

`data_quality.csv` را مرحله‌ی `outputs` از خود `cases.csv` می‌سازد (بخش
۴.۸.۴ قرارداد). برای هر ستون: درصد خالی، تعداد یکتا، دامنه و پرتکرارترین
مقادیر. این جدول نقطه‌ی شروع است: هر ادعای بعدی باید با همین تصویر
هم‌خوان باشد — مثلاً ستون `duration_minutes` همان‌جا نشان می‌دهد چه سهمی
از پرونده‌ها باز مانده‌اند.

In [ ]:
quality = pd.read_csv(QUALITY)
quality[["column", "kind", "rows", "empty_pct", "unique_count", "min", "max", "top_values"]]

## ۳) توزیع مدت بسته‌شدن — آیا «میانگین» عدد درستی است؟

سؤال: یک پرونده‌ی صوتی معمولاً چقدر طول می‌کشد بسته شود؟

چرا مهم است: اگر توزیع دم‌سنگین باشد، میانگین با چند پرونده‌ی چندساله
کشیده می‌شود و گزارش با «میانگین ۳۰ روز» گمراه‌کننده می‌شود؛ آن‌وقت
میانه و صدک‌ها معیار درست‌اند.

پرونده‌های **باز** در محاسبه‌ی مدت نمی‌آیند (مدت ندارند)، اما سهمشان
کنار نتیجه گزارش می‌شود — حذف بی‌صدای آن‌ها همه‌ی ادعاهای مدت را سوگیر
می‌کند.

In [ ]:
summary = eda.duration_summary(frame)
print(f"cases        : {summary['cases']:,}")
print(f"closed       : {summary['closed_cases']:,}  ({1 - summary['open_share']:.2%})")
print(f"open         : {summary['open_cases']:,}  ({summary['open_share']:.2%})")
print(f"mean (closed): {summary['mean_days']:.1f} روز")
print(f"median       : {summary['median_days']:.2f} روز")
print(f"p90 / p99    : {summary['p90_days']:.1f} / {summary['p99_days']:.1f} روز")
print(f"max          : {summary['max_days']:.0f} روز")
print(f"mean ÷ median: {summary['mean_days'] / summary['median_days']:.1f}×")

In [ ]:
buckets = eda.duration_buckets(frame)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(buckets["bucket"], buckets["cases"], color="#3b6ea5")
ax.set_yscale("log")
ax.set_xlabel("time to close")
ax.set_ylabel("cases (log scale)")
ax.set_title("Chart 1 — closure duration: heavy tail, no single scale")
for rect, share in zip(bars, buckets["share"]):
    ax.annotate(f"{share:.1%}", (rect.get_x() + rect.get_width() / 2, rect.get_height()),
                ha="center", va="bottom", fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
fast_share = float(buckets.loc[buckets["bucket"].isin(["<1h", "1-6h"]), "share"].sum())
slow_share = float(buckets.loc[buckets["bucket"].isin(["90-365d", ">365d"]), "share"].sum())
skew = summary["mean_days"] / summary["median_days"]
print("نتیجه (Chart 1):")
print(f"  • {fast_share:.1%} پرونده‌های بسته‌شده در کمتر از ۶ ساعت بسته می‌شوند؛"
      f" {slow_share:.1%} بیشتر از ۹۰ روز طول می‌کشند.")
if skew >= 2:
    print(f"  • میانگین {summary['mean_days']:.1f} روز در برابر میانه {summary['median_days']:.2f} روز"
          f" ({skew:.0f}×) — توزیع دم‌سنگین است؛ در گزارش میانه و p90 نقل می‌شود، نه میانگین.")
else:
    print(f"  • میانگین {summary['mean_days']:.1f} روز و میانه {summary['median_days']:.2f} روز"
          f" ({skew:.1f}×) نزدیک‌اند؛ در این داده میانگین گمراه‌کننده نیست، ولی صدک‌ها همچنان گزارش می‌شوند.")

## ۴) پرت‌ها (outliers) و تعیین تکلیفشان

**تصمیم: پرت‌ها حذف نمی‌شوند.** یک پرونده‌ی دوساله یک واقعیت فرآیندی است
(بیروکراتیک یا رهاشده)، نه خطای ثبت. حذف آن سؤال «چرا این‌قدر طول کشید؟»
را از داده پاک می‌کند.

اما برای *توصیف*، میانگین را روی سقف یک سال می‌بندیم تا چند رکورد کل
تصویر را نکشد؛ میانه و صدک‌ها را روی داده‌ی کامل گزارش می‌کنیم. سقف یک
عدد اعلام‌شده است («یک سال»)، نه یک برش پنهان.

In [ ]:
outliers = eda.outlier_summary(frame)
print(f"cap                     : {outliers['cap_days']:.0f} روز")
print(f"cases beyond cap        : {outliers['beyond_cap_cases']:,.0f} "
      f"({outliers['beyond_cap_share']:.3%} از پرونده‌های بسته)")
print(f"mean (all closed)       : {outliers['mean_days_all']:.1f} روز")
print(f"mean (capped at 1 year) : {outliers['mean_days_within_cap']:.1f} روز")
print(f"median / max            : {outliers['median_days']:.2f} / {outliers['max_days']:.0f} روز")

In [ ]:
durations = frame["duration_minutes"].dropna().sort_values()
ecdf = np.arange(1, len(durations) + 1) / len(durations)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(durations, ecdf, color="#3b6ea5", linewidth=1.6, label="ECDF (closed cases)")
for quantile, colour in ((0.50, "#6b7280"), (0.90, "#c2410c"), (0.99, "#b91c1c")):
    value = float(durations.quantile(quantile))
    ax.axvline(value, color=colour, linestyle="--", linewidth=1,
               label=f"p{int(quantile * 100)} = {value / 1440:.1f} d")
ax.axvline(eda.OUTLIER_CAP_DAYS * 1440, color="black", linestyle=":", linewidth=1,
           label="outlier cap = 1 year")
ax.set_xscale("log")
ax.set_xlabel("closure duration (minutes, log scale)")
ax.set_ylabel("share of closed cases")
ax.set_title("Chart 2 — ECDF: half the cases close fast, a long tail never does")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
print("نتیجه (Chart 2):")
print(f"  • نیمی از پرونده‌ها تا {summary['median_days']:.2f} روز بسته می‌شوند،"
      f" اما ۱۰٪ بالایی از {summary['p90_days']:.1f} روز عبور می‌کنند")
print(f"    و ۱٪ آخر تا {summary['p99_days']:.0f} روز می‌روند؛ فاصله‌ی p50 تا p99 "
      f"{summary['p99_days'] / max(summary['median_days'], 1e-9):.0f}× است.")
print(f"  • تکلیف پرت‌ها: نگه‌داشتن (میانگین روی سقف یک سال بسته می‌شود؛"
      f" میانگین سقف‌دار {outliers['mean_days_within_cap']:.1f} روز در برابر"
      f" {outliers['mean_days_all']:.1f} روز بدون سقف).")

## ۵) پرونده‌های باز، به تفکیک نوع شکایت

سؤال: آیا «باز ماندن» بین شش نوع شکایت یکسان است؟

چرا مهم است: اگر یک نوع شکایت مثلاً ۶۰٪ باز داشته باشد، هر مقایسه‌ی
مدت بین انواع به نفع آن نوع سوگیر می‌شود — چون سخت‌ترین پرونده‌هایش
اصلاً در میانگین/میانه نیستند. مقایسه‌ی مدت فقط وقتی منصفانه است که
سهم بازها کنارش باشد.

In [ ]:
by_type = eda.open_share_by_type(frame)

fig, ax = plt.subplots(figsize=(9, 4))
labels = by_type.index[::-1]
values = by_type["open_share"].to_numpy()[::-1]
bars = ax.barh(labels, values, color="#b45309")
ax.set_xlabel("share of cases still open")
ax.set_title("Chart 3 — open share differs by complaint type")
for rect, value, cases in zip(bars, values, by_type["cases"].to_numpy()[::-1]):
    ax.annotate(f"{value:.1%} of {cases:,}", (rect.get_width(), rect.get_y() + rect.get_height() / 2),
                va="center", ha="left", fontsize=8)
ax.set_xlim(0, max(values) * 1.35 if len(values) else 1)
fig.tight_layout()
plt.show()
by_type

In [ ]:
widest = by_type["open_share"]
print("نتیجه (Chart 3):")
print(f"  • سهم پرونده‌های باز بین «{widest.index[0]}» ({widest.iloc[0]:.1%})"
      f" و «{widest.index[-1]}» ({widest.iloc[-1]:.1%}) تغییر می‌کند"
      f" — یعنی {widest.iloc[0] / max(widest.iloc[-1], 1e-9):.1f}× اختلاف.")
print("  • پس هر مقایسه‌ی مدت بین انواع، باید کنار سهم بازها خوانده شود؛"
      " در غیر این صورت نوعی که پرونده‌های سختش باز مانده‌اند «سریع‌تر» به نظر می‌رسد.")

## ۶) ترکیب انواع و الگوی ماهانه

سؤال: حجم کل از چه چیزی ساخته می‌شود و آیا در طول ۱۸ ماه تغییر می‌کند؟

چرا مهم است: اگر یک نوع شکایت اکثریت داده را بسازد، هر «میانگین شهر»
عملاً توصیف همان نوع است. و اگر الگوی فصلی وجود داشته باشد، پنجره‌ی
۱۸ماهه نباید با یک ماه نمونه مقایسه شود.

In [ ]:
mix = eda.type_mix(frame)
monthly = eda.monthly_volume(frame)

fig, ax = plt.subplots(figsize=(10, 4.2))
for complaint_type in monthly.columns:
    ax.plot(monthly.index, monthly[complaint_type].to_numpy(),
            linewidth=1.6, label=complaint_type)
ax.set_ylabel("cases per month")
ax.set_title("Chart 4 — monthly volume by complaint type")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.legend(fontsize=7, ncol=2, loc="upper left")
fig.tight_layout()
plt.show()
mix

In [ ]:
seasonal_peak, seasonal_low = monthly.sum(axis=1).idxmax(), monthly.sum(axis=1).idxmin()
print("نتیجه (Chart 4):")
print(f"  • «{mix.index[0]}» به‌تنهایی {mix['share'].iloc[0]:.1%} از کل پرونده‌هاست"
      f" (۵ نوع دیگر روی هم {1 - mix['share'].iloc[0]:.1%}).")
print(f"  • شلوغ‌ترین ماه {seasonal_peak} با {int(monthly.sum(axis=1).max()):,} پرونده و"
      f" خلوت‌ترین ماه {seasonal_low} با {int(monthly.sum(axis=1).min()):,} پرونده است"
      f" ({monthly.sum(axis=1).max() / monthly.sum(axis=1).min():.1f}×).")

## ۷) شمارش خام در برابر نرخ به‌ازای جمعیت

سؤال: آیا «بیشترین پرونده» یعنی «بیشترین شکایت به ازای هر نفر»؟

چرا مهم است: این همان دلیلی است که منبع دوم (جمعیت) وارد کارخانه شد.
اگر رتبه‌بندی خام و رتبه‌بندی نرمال‌شده یکی باشند، جدول نرخ فقط تزئین
است؛ اگر فرق کنند، هر گزارشی که فقط شمارش خام را ببیند نتیجه‌ی غلط
می‌دهد.

نکته‌ی روش: boroughهای بدون جمعیت مرجع، نرخ خالی می‌گیرند — هیچ
مخرج‌کننده‌ای ساخته نمی‌شود.

In [ ]:
rates = eda.borough_rates(frame, population)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
axes[0].barh(rates.index[::-1], rates["cases"].to_numpy()[::-1], color="#3b6ea5")
axes[0].set_title("raw cases")
axes[1].barh(rates.index[::-1], rates["rate_per_100k"].to_numpy()[::-1], color="#0f766e")
axes[1].set_title("cases per 100k residents")
for ax, column in zip(axes, ("cases", "rate_per_100k")):
    span = ax.get_xlim()[1]
    for index, (_, row) in enumerate(rates.iloc[::-1].iterrows()):
        value = row[column]
        if pd.isna(value):
            ax.annotate("no denominator", (span * 0.02, index), va="center",
                        ha="left", fontsize=7, color="#b91c1c")
        else:
            ax.annotate(f"{value:,.0f}", (value, index), va="center",
                        ha="right", fontsize=8, color="white")
    ax.set_xlabel(column)
fig.suptitle("Chart 5 — raw counts vs population-normalised rates", y=1.02)
fig.tight_layout()
plt.show()
rates

In [ ]:
top_count = rates["cases"].idxmax()
top_rate = rates["rate_per_100k"].idxmax()
print("نتیجه (Chart 5):")
print(f"  • بیشترین شمارش خام: {top_count} ({rates.loc[top_count, 'cases']:,.0f} پرونده)")
print(f"  • بیشترین نرخ سرانه: {top_rate} ({rates.loc[top_rate, 'rate_per_100k']:,.0f} به ازای ۱۰۰٬۰۰۰ نفر)")
if top_count == top_rate:
    print("  • رتبه‌بندی خام و سرانه یکی درآمد؛ در این داده نرمال‌سازی رتبه‌ها را جابه‌جا نکرد.")
else:
    print(f"  • رتبه‌بندی عوض شد: {top_rate} با وجود شمارش کمتر، سرانه بالاتری دارد؛"
          " گزارش بر پایه‌ی شمارش خام نتیجه‌ی دیگری می‌داد.")

## ۸) رابطه‌ی متغیرهای عددی و مقیاس ZIP

دو سؤال: آیا مدت بسته‌شدن با جمعیت منطقه رابطه دارد؟ و آیا حجم پرونده‌ی
هر ZIP فقط بازتاب جمعیت آن است؟

روش: همبستگی پیرسون روی ستون‌های عددی، و یک برازش کمترین‌مربع ساده
(حجم = a + b × جمعیت) که باقی‌مانده‌ی هر ZIP را نسبت به مقدار انتظاری
نشان می‌دهد. باقی‌مانده‌ی مثبت بزرگ یعنی «اینجا بیشتر از حد انتظار شکایت
می‌شود» — کاندیدای بررسی میدانی، نه لزوماً خطای داده.

In [ ]:
correlations = eda.numeric_relationships(frame)
zip_table = eda.zip_volume_vs_population(frame, population)

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.scatter(zip_table["population"], zip_table["cases"], s=22, alpha=0.7,
           color="#3b6ea5", label="ZIP")
order = zip_table.sort_values("population")
ax.plot(order["population"], order["fitted"], color="#b91c1c", linewidth=1.4,
        label="least-squares fit")
worst = zip_table["residual_share"].idxmax()
ax.annotate(f"ZIP {worst}: {zip_table.loc[worst, 'residual_share']:+.0%} vs expected",
            (zip_table.loc[worst, "population"], zip_table.loc[worst, "cases"]),
            textcoords="offset points", xytext=(12, -12), fontsize=8, color="#b91c1c")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("ZIP population (log scale)")
ax.set_ylabel("cases in window (log scale)")
ax.set_title("Chart 6 — volume tracks population, with real residuals")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
correlations

In [ ]:
print("نتیجه (Chart 6):")
print(f"  • همبستگی مدت با جمعیت: {correlations.loc['duration_days', 'population']:+.3f}"
      f" و با ساعت ثبت: {correlations.loc['duration_days', 'created_hour']:+.3f}"
      " — تقریباً هیچ رابطه‌ی خطی بین مدت و اندازه‌ی منطقه نیست.")
residual = zip_table["residual_share"]
print(f"  • باقی‌مانده‌ی ZIPها بین {residual.min():+.0%} و {residual.max():+.0%} است؛"
      f" بدترین مورد ZIP {worst} با {zip_table.loc[worst, 'cases']:,.0f} پرونده در برابر"
      f" {zip_table.loc[worst, 'fitted']:,.0f} انتظار.")
print(f"  • یعنی حجم هر ZIP با جمعیت توضیح داده می‌شود ولی نه کامل؛"
      f" پراکندگی باقی‌مانده ({residual.std():.0%}) خودش یافته است، نه نویز.")

## ۹) اثر بسته‌شدن خودکار: چه ساعتی پرونده‌ها سریع بسته می‌شوند؟

فرضیه: سبد «کمتر از یک ساعت» در Chart 1 تصادفی نیست؛ بخشی از پرونده‌ها
در ساعات خاصی (نزدیک پایان شیفت یا در پردازش دسته‌ای شبانه) تقریباً بلافاصله
بسته می‌شوند.

روش: سهم پرونده‌هایی که در ≤۶۰ دقیقه بسته شده‌اند، به تفکیک ساعت ثبت.
اگر الگو تخت باشد، سبد سریع فقط «کار سریع» است؛ اگر قله داشته باشد،
یعنی *زمان ثبت* روی مدت اثر می‌گذارد و هر تحلیل مدتی باید این را بداند.

In [ ]:
speed = eda.closure_speed_by_hour(frame, within_minutes=60)

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(speed.index, speed["within_share"], color="#7c3aed")
ax.set_xlabel("hour of day the case was created")
ax.set_ylabel("share closed within 60 minutes")
ax.set_title("Chart 7 — fast closures cluster by hour of the day")
ax.set_xticks(range(0, 24, 2))
fig.tight_layout()
plt.show()
speed.describe().round(3)

In [ ]:
peak_hour = int(speed["within_share"].idxmax())
quiet_hour = int(speed["within_share"].idxmin())
print("نتیجه (Chart 7):")
print(f"  • بیشترین سهم بسته‌شدن سریع در ساعت {peak_hour:02d}:00 است"
      f" ({speed.loc[peak_hour, 'within_share']:.1%}) و کم‌ترین در ساعت {quiet_hour:02d}:00"
      f" ({speed.loc[quiet_hour, 'within_share']:.1%}).")
ratio = speed["within_share"].max() / max(speed["within_share"].min(), 1e-9)
print(f"  • یعنی زمان ثبت، احتمال بسته‌شدن سریع را تا {ratio:.1f}× تغییر می‌دهد؛"
      " پس در تفسیر مدل‌های سریع باید ساعت را کنترل کرد.")

## ۱۰) یافته‌های غیربدیهی

سه (و بیشتر) چیزی که پیش از دیدن داده نمی‌دانستیم. هر مورد از سلول‌های
بالا محاسبه می‌شود، نه از حافظه.

In [ ]:
metrics = eda.finding_metrics(frame, population)

print("یافته ۱ — «میانگین مدت» عدد گمراه‌کننده‌ای است:")
print(f"   میانگین {metrics['mean_days']:.1f} روز در برابر میانه {metrics['median_days']:.2f} روز"
      f" ({metrics['mean_to_median']:.0f}×). دلیلش دم دم‌سنگین است، نه کندی معمول کار.")
print()
print("یافته ۲ — سهم پرونده‌های باز بین انواع شکایت یکسان نیست:")
print(f"   «{metrics['most_open_type']}» {metrics['most_open_share']:.1%} باز،"
      f" «{metrics['least_open_type']}» {metrics['least_open_share']:.1%}.")
print(f"   در نتیجه مقایسه‌ی خام مدت بین انواع، نوعِ با پرونده‌های بازِ بیشتر را «سریع‌تر» نشان می‌دهد.")
print()
print("یافته ۳ — یک نوع شکایت تقریباً کل داده را می‌سازد:")
print(f"   «{metrics['top_type']}» {metrics['top_type_share']:.1%} پرونده‌هاست؛"
      " هر «میانگین شهر» عملاً توصیف همین نوع است.")
print()
print("یافته ۴ — شمارش خام و نرخ سرانه یک رتبه‌بندی نمی‌دهند:")
print(f"   بیشترین شمارش: {metrics['top_count_borough']} | بیشترین نرخ سرانه: {metrics['top_rate_borough']}"
      f" ({metrics['rate_leader_over_count_leader']:.1f}× نرخِ صدر جدول شمارش).")
print()
print("یافته ۵ — حجم ZIPها فقط تابع جمعیت نیست:")
print(f"   ZIP {metrics['worst_residual_zip']} با {metrics['worst_residual_cases']:,.0f} پرونده،"
      f" {metrics['worst_residual_share']:+.0%} بیشتر از انتظار برازش جمعیتی.")
print()
print("یافته ۶ — ساعت ثبت روی سرعت بستن اثر دارد:")
print(f"   ساعت {metrics['peak_fast_hour']:02d}:00 با {metrics['peak_fast_hour_share']:.1%}"
      f" بسته‌شدن زیر یک ساعت، در برابر {metrics['quiet_fast_hour_share']:.1%}"
      f" در ساعت {metrics['quiet_fast_hour']:02d}:00 — اثر پردازش دسته‌ای/شیفتی.")

## ۱۱) محدودیت‌ها و آنچه این تحلیل نمی‌گوید

* **علیت ندارد.** Chart 7 نشان می‌دهد ساعت ثبت و سرعت بستن همراه‌اند؛
  درباره‌ی علت (شیفت، اتوماسیون، بار کاری) چیزی ثابت نمی‌کند.
* **پنجره ثابت است.** هیچ ادعایی درباره‌ی روند بلندمدت نمی‌کنیم؛ ۱۸ ماه
  برای «روند» کوتاه است و پنجره‌ی متحرک (`--since-last-run`) عمداً
  پنجره‌های کوتاه می‌سازد که با این تحلیل قاطی نشوند.
* **جمعیت، مخرج ساده است.** نرخ‌ها پرونده به ازای ساکن است، نه به ازای
  خانوار یا کاربر؛ ZIPهای بدون جمعیت مرجع نرخ خالی دارند و در رتبه‌بندی
  نمی‌آیند.
* **کیفیت داده، پیش‌فرض است.** اگر منبع بالادست ستونی را حذف کند، کارخانه
  با خطا می‌ایستد (بخش ۴.۸.۲) و این نوتبوک اصلاً اجرا نمی‌شود — که همان
  رفتار درست است.
* **زمان اجرا:** همه‌ی محاسبات این نوتبوک یک‌بار از `cases.csv` می‌خوانند
  و روی یک لپ‌تاپ معمولی در چند ده ثانیه تمام می‌شوند (سقف قرارداد:
  ۱۵ دقیقه).